In [ ]:
code = 'B120_D'
pickle_path = 'P:/PGC Data/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    queue_excel = pd.read_excel(Path(parameter_path).parent / "Queue.xlsx", None)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def xlsx_to_dict(queue_excel):
    result = {}
    for sheet, df in queue_excel.items():
        groups = {}
        for g in range(df.shape[1] // 4):
            sub = df.iloc[:, g*4:g*4+4].dropna(how="all")
            sub.columns = ["entry", "om", "sl", "ut_sl"]
            rows = {}
            for i, r in enumerate(sub.to_dict("records"), 1):
                rows[i] = {
                    "entry": r["entry"],
                    "sl":    float(r["sl"]),
                    "ut_sl": r["ut_sl"],
                    "om":    r["om"],
                }
            groups[f"group_{g+1}"] = rows
        result[sheet] = groups
    return result

queue_groups = xlsx_to_dict(queue_excel)

In [ ]:
def b120(bt, start_time, end_time, orderside, method, sl, ut_sl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, start_dt, om=om)
        if ce_scrip is None: return None
        
        from_candle_close = True if method == 'CC' else False

        entry_time = start_dt
        ce_open, ce_high, ce_low, ce_close, ce_sl_price, ce_sl_flag, _, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sl, with_ohlc=True, orderside=orderside, from_candle_close=from_candle_close)
        pe_open, pe_high, pe_low, pe_close, pe_sl_price, pe_sl_flag, _, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sl, with_ohlc=True, orderside=orderside, from_candle_close=from_candle_close)
        ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
        pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m

        ut_scrip, ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, ut_sl_time, ut_pnl = '', '', '', '', '', '', False, '', 0
        ut_sl = ut_sl if str(ut_sl) == 'TTC' else float(ut_sl)
        B_PL, TT_PL_at_SL, UT_PL_at_SL = 0, 0, 0
        
        if ce_sl_time < pe_sl_time:
            pe_sl_time = end_dt_1m

            ut_sl_price = pe_price if str(ut_sl) == 'TTC' else None
            ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, _, _, ut_sl_time, ut_pnl = bt.sl_check_single_leg(ce_sl_time, end_dt, pe_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close)

            if ut_open:
                ut_scrip = pe_scrip
                TT_PL_at_SL = ce_pnl
                UT_PL_at_SL = pe_price - ut_open - bt.Cal_slipage(pe_price)
                
                if (str(ut_sl) == 'TTC') and (ut_open > ut_sl_price):
                    ut_sl_price = pe_sl_price
                    ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, _, _, ut_sl_time, ut_pnl = bt.sl_check_single_leg(ce_sl_time, end_dt, pe_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close)
            else:
                B_PL = ce_pnl + pe_pnl

        elif pe_sl_time < ce_sl_time:
            ce_sl_time = end_dt_1m

            ut_sl_price = ce_price if str(ut_sl) == 'TTC' else None
            ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, _, _, ut_sl_time, ut_pnl = bt.sl_check_single_leg(pe_sl_time, end_dt, ce_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close)

            if ut_open:
                ut_scrip = ce_scrip
                TT_PL_at_SL = pe_pnl
                UT_PL_at_SL = ce_price - ut_open - bt.Cal_slipage(ce_price)
                
                if (str(ut_sl) == 'TTC') and (ut_open > ut_sl_price):
                    ut_sl_price = ce_sl_price
                    ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, _, _, ut_sl_time, ut_pnl = bt.sl_check_single_leg(pe_sl_time, end_dt, ce_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close)
            else:
                B_PL = ce_pnl + pe_pnl

        else:
            B_PL = ce_pnl + pe_pnl
        
        if ut_open:
            b120_sl_time = ut_sl_time
        else:
            b120_sl_time = max(ce_sl_time, pe_sl_time)

        b120_sl_time = '' if b120_sl_time == end_dt_1m else b120_sl_time

        total_pnl = B_PL + TT_PL_at_SL + UT_PL_at_SL + ut_pnl
        return [entry_time, ce_scrip, pe_scrip, b120_sl_time, total_pnl]
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, sl, ut_sl, om])
        return

In [ ]:
def b120_d(bt, start_time, end_time, orderside, method, lots, index_queue_groups):
    
    live_trades = {}
    current_trade = 0
    max_trade = lots
    total_trades = 0
    current_time = start_time
    total_pnl = 0
    
    entry_time = start_time
    future_price = bt.future_data['close'].iloc[0]
    
    logs = [code, bt.index, start_time, end_time, orderside, method, lots, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time, future_price]
    
    for group, params in index_queue_groups.items():

        for idx, param in params.items():

            current_time = param['entry']

            if current_trade < max_trade:
                b120_output = b120(bt, param['entry'], end_time, orderside, method, param['sl'], param['ut_sl'], param['om'])
                if b120_output:
                    live_trades[f"{group}-{idx}"] = dict(zip(['entry', 'ce_scrip', 'pe_scrip', 'exit', 'pnl'], b120_output))
                    total_pnl += live_trades[f"{group}-{idx}"]['pnl']
                    current_trade += 1
                    total_trades += 1

        stop_out_trades = {k: v for k, v in live_trades.items() if v['exit'] and v['exit'].time() < current_time}
        live_trades = {k: v for k, v in live_trades.items() if not v['exit'] or v['exit'].time() >= current_time}
        current_trade = len(live_trades)
    
    avg_pnl = total_pnl/total_trades
    
    logs.append(avg_pnl)
    logs.append(total_pnl)

    return logs

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)

            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_Lots/Date/Day/DTE/EntryTime/Future/Avg.PNL/Total.PNL').split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    index_queue_groups = queue_groups[index]
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [b120_d(bt, row.entry_time, row.exit_time, row.orderside, row.method, row.lots, index_queue_groups) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))